# Canal Cero — Agente de Optimización Publicitaria con MMM Bayesiano
### Entregable Final · Roadmap 10 pasos · UAI · Programa de IA

**Cliente:** Tienda Copec  
**Idea (Taller 3 elegida):** Agente que responde preguntas sobre performance de medios y optimización de presupuesto, apoyado en un **Marketing Mix Model (MMM)** bayesiano y en documentos de negocio (RAG).

Este notebook implementa los **10 pasos** del roadmap:

| Fase | Paso | Contenido |
|------|------|-----------|
| **1. Fundamentos** | 1 | Documentos de negocio (5 .txt) |
| | 2 | Embeddings + Vector DB (Chroma) con metadatos |
| | 3 | Agente Orquestador (system prompt + ruteo) |
| | 4 | Workers: RAG, SQL, MMM (con retry y manejo de errores) |
| | 5 | Agente Fiscalizador (auditor de la respuesta) |
| | 6 | Búsqueda semántica KNN (k=5) + 10 consultas reales + precisión |
| **2. Producción** | 7 | SQL: tabla `interactions` (trazabilidad) |
| | 8 | Estructura GitHub |
| | 9 | Dockerfile + Cloud Run + Secret Manager |
| | 10 | Deploy & Go Live + reporte (latencia, costo/1000, mejoras) |

> **Nota de diseño:** el PPT del proyecto propone implementar el flujo en **n8n + Pinecone + PyMC**. Para que la evaluación sea reproducible en Colab/Jupyter sin infraestructura externa, aquí se implementa la **misma arquitectura lógica en Python**, usando **Chroma** (en vez de Pinecone) y un **MMM ligero local** (en vez del microservicio PyMC). El mapeo n8n→Python se documenta en cada paso y los workflows n8n se incluyen en `/workflows` del repo.


## 0 · Configuración del entorno

Instala dependencias y define las API keys. En Colab puedes guardar `OPENAI_API_KEY` y `ANTHROPIC_API_KEY` en *Secrets* (🔑 panel izquierdo) o asignarlas abajo.

El notebook funciona en **dos modos**:
- **Modo OpenAI/Claude** (recomendado para la entrega): usa `text-embedding-3-small` y un LLM real.
- **Modo offline** (sin API key): usa embeddings locales (`sentence-transformers`) y un LLM simulado determinista, para que el notebook **corra completo aunque no haya créditos**. Se activa solo si no hay keys.

In [23]:
!pip install -q openai anthropic chromadb sentence-transformers pandas numpy scikit-learn tabulate 2>/dev/null
print("Dependencias instaladas")

Dependencias instaladas


In [24]:
import os, json, time, uuid, sqlite3, re, math, datetime
import numpy as np, pandas as pd
from google.colab import userdata

# === API KEYS ===
# Conectamos con el panel de Secrets de Colab de forma segura
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

os.environ.setdefault("OPENAI_API_KEY", os.environ.get("OPENAI_API_KEY",""))
os.environ.setdefault("ANTHROPIC_API_KEY", os.environ.get("ANTHROPIC_API_KEY",""))

USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
USE_CLAUDE = bool(os.environ.get("ANTHROPIC_API_KEY"))
OFFLINE = not (USE_OPENAI or USE_CLAUDE)
print(f"OpenAI={'ON' if USE_OPENAI else 'off'} | Claude={'ON' if USE_CLAUDE else 'off'} | OFFLINE={OFFLINE}")

OpenAI=ON | Claude=off | OFFLINE=False


### Rutas de datos
Sube al notebook la carpeta `docs/` (5 documentos) y `data/mmm_daily.csv`, `data/mmm_outputs.csv`. En Colab puedes clonar el repo o usar el panel de *Files*. Aquí se asume la estructura del repositorio.

In [25]:
DOCS_DIR = "docs"
DATA_DIR = "data"

# Si corres en Colab y subiste un zip, descomprime aqui. Verificacion:
import pathlib
for p in [DOCS_DIR, DATA_DIR]:
    if not pathlib.Path(p).exists():
        print(f"[AVISO] No existe {p}/ . Sube los archivos del repo (docs/ y data/).")
print("docs:", os.listdir(DOCS_DIR) if pathlib.Path(DOCS_DIR).exists() else "FALTA")
print("data:", os.listdir(DATA_DIR) if pathlib.Path(DATA_DIR).exists() else "FALTA")

docs: ['diccionario_datos.txt', 'guia_meridian.txt', 'calendario_eventos.txt', 'metodologia_mmm.txt', 'glosario_campanas.txt']
data: ['budget_optimization.csv', 'mmm_outputs.csv', 'mmm_daily.csv']


---
## Paso 1 · Documentos de negocio

Base de conocimiento del agente: **5 documentos** representativos del caso Tienda Copec.

| Documento | Contenido | Preguntas que habilita |
|-----------|-----------|------------------------|
| `diccionario_datos.txt` | Esquema de `mmm_daily.csv`, unidades, calidad de datos | "¿qué significa la columna X?", "¿en qué unidad está la inversión?" |
| `metodologia_mmm.txt` | Qué es MMM bayesiano, adstock, saturación, ROI vs ROAS | "¿qué es adstock?", "¿por qué el ROI del MMM difiere del ROAS?" |
| `glosario_campanas.txt` | Tipos de campaña, métricas, canales, estructura PDM | "¿qué es una campaña AON?", "¿qué meta de ROAS pide Copec?" |
| `calendario_eventos.txt` | Eventos comerciales (Black Friday, CyberDay, etc.) | "¿qué explica el pico de venta de noviembre?" |
| `guia_meridian.txt` | Implementación MMM (Meridian/PyMC), insumos y salidas | "¿qué insumos necesita el modelo?", "¿cómo leo el ROI incremental?" |

**Preguntas de negocio que el sistema debe responder** (las usamos en el Paso 6):
1. ¿Cuál fue la inversión total en Meta en el periodo? *(SQL)*
2. ¿Qué canal tiene mejor ROI incremental según el MMM? *(SQL/MMM)*
3. ¿Qué es el adstock y por qué hay venta sin gasto el mismo día? *(RAG)*
4. ¿Qué meta de ROAS acordó Copec? *(RAG)*
5. ¿Cuánto presupuesto conviene mover de TikTok a Google? *(MMM)*
... (set completo en el Paso 6)

In [26]:
def load_docs(docs_dir):
    docs = []
    for fn in sorted(os.listdir(docs_dir)):
        if fn.endswith(".txt"):
            text = open(os.path.join(docs_dir, fn), encoding="utf-8").read()
            docs.append({"source": fn, "text": text,
                         "fecha": "2026-05-17"})
    return docs

documents = load_docs(DOCS_DIR)
print(f"{len(documents)} documentos cargados:")
for d in documents:
    print(f"  - {d['source']}: {len(d['text'])} chars")

5 documentos cargados:
  - calendario_eventos.txt: 1810 chars
  - diccionario_datos.txt: 3495 chars
  - glosario_campanas.txt: 2366 chars
  - guia_meridian.txt: 2329 chars
  - metodologia_mmm.txt: 2841 chars


---
## Paso 2 · Embeddings y Base de Datos Vectorial

**Chunking** (500 chars / 50 de solape, igual que el Token Splitter del PPT) → **embeddings** → **Chroma** con metadatos `fuente`, `fecha`, `chunk_id`.

*Mapeo n8n:* `Read File → Token Splitter (500/50) → Embeddings OpenAI text-embedding-3-small → Pinecone`. Aquí: Chroma local con la misma config.

In [27]:
def chunk_text(text, size=500, overlap=50):
    chunks, start = [], 0
    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start = end - overlap
        if start <= 0: break
    return [c.strip() for c in chunks if c.strip()]

all_chunks = []
for d in documents:
    for i, ch in enumerate(chunk_text(d["text"])):
        all_chunks.append({
            "id": f"{d['source']}::ch{i}",
            "text": ch,
            "metadata": {"fuente": d["source"], "fecha": d["fecha"], "chunk_id": i}
        })
print(f"Total chunks: {len(all_chunks)}")

Total chunks: 32


In [28]:
# --- Funcion de embeddings: OpenAI si hay key, si no sentence-transformers ---
if USE_OPENAI:
    from openai import OpenAI
    _oai = OpenAI()
    def embed(texts):
        r = _oai.embeddings.create(model="text-embedding-3-small", input=texts)
        return [e.embedding for e in r.data]
    EMB_NAME = "text-embedding-3-small (OpenAI)"
else:
    from sentence_transformers import SentenceTransformer
    _st = SentenceTransformer("all-MiniLM-L6-v2")
    def embed(texts):
        return _st.encode(texts, normalize_embeddings=True).tolist()
    EMB_NAME = "all-MiniLM-L6-v2 (local)"
print("Modelo de embeddings:", EMB_NAME)

Modelo de embeddings: text-embedding-3-small (OpenAI)


In [29]:
import chromadb
client = chromadb.Client()
try: client.delete_collection("canal_cero_docs")
except: pass
collection = client.create_collection("canal_cero_docs", metadata={"hnsw:space":"cosine"})

# Indexar en lotes
B = 100
for i in range(0, len(all_chunks), B):
    batch = all_chunks[i:i+B]
    collection.add(
        ids=[c["id"] for c in batch],
        embeddings=embed([c["text"] for c in batch]),
        documents=[c["text"] for c in batch],
        metadatas=[c["metadata"] for c in batch],
    )
print(f"Indexados {collection.count()} chunks en Chroma (coleccion canal_cero_docs)")

Indexados 32 chunks en Chroma (coleccion canal_cero_docs)


---
## Paso 6 · Búsqueda semántica KNN (retriever k=5)

Definimos el retriever **antes** de los workers porque el Worker RAG lo usa. Más abajo (final del Paso 6) corremos las **10 consultas reales** y medimos **precisión@k**.

In [30]:
K = 5
def retrieve(query, k=K):
    res = collection.query(query_embeddings=embed([query]), n_results=k)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        hits.append({"text": doc, "fuente": meta["fuente"], "fecha": meta["fecha"],
                     "score": round(1 - dist, 4)})
    return hits

# prueba rapida
for h in retrieve("que es el adstock"):
    print(f'[{h["score"]}] {h["fuente"]}: {h["text"][:70]}...')

[0.517] metodologia_mmm.txt: bre, lo que es honesto para decisiones de presupuesto.
- Maneja bien l...
[0.4819] diccionario_datos.txt: xport
   original; antes de modelar conviene normalizar a USD reales. ...
[0.4428] glosario_campanas.txt: , todo el mes, mantienen demanda.
BIG SALE / BLACK SALE / BLACK FRIDAY...
[0.4085] guia_meridian.txt: ularidad diaria, alineados por fecha. Fuente: data/mmm_daily.csv.

3. ...
[0.4084] diccionario_datos.txt: RANGO POR FUENTE ORIGINAL
- meta_ads:        2023-01-01 a 2024-08-22 (...


### Helper de LLM (orquestador, workers y fiscalizador lo usan)
Usa Claude si hay key, OpenAI si hay, o un **stub determinista** offline para que el notebook corra completo.

In [31]:
def llm(system, user, max_tokens=700, temperature=0):
    """Devuelve texto del LLM. Soporta Claude, OpenAI o stub offline."""
    if USE_CLAUDE:
        import anthropic
        c = anthropic.Anthropic()
        m = c.messages.create(model="claude-sonnet-4-5-20250929", max_tokens=max_tokens,
            temperature=temperature, system=system,
            messages=[{"role":"user","content":user}])
        return "".join(b.text for b in m.content if b.type=="text")
    if USE_OPENAI:
        from openai import OpenAI
        r = OpenAI().chat.completions.create(model="gpt-4o-mini", max_tokens=max_tokens,
            temperature=temperature,
            messages=[{"role":"system","content":system},{"role":"user","content":user}])
        return r.choices[0].message.content
    # OFFLINE stub: heuristica simple para no romper el flujo
    return f"[OFFLINE-LLM] No hay API key. Resumen del contexto:\n{user[:400]}"

---
## Paso 7 · SQL para registros (lo definimos temprano: lo usan los workers)

Creamos una base **SQLite** con:
- `mmm_daily`: datos diarios reales (fuente del Worker SQL).
- `mmm_outputs`: salidas del MMM (ROI, contribución, saturación).
- `budget_optimization`: recomendación de reasignación.
- `interactions`: **trazabilidad** (session_id, timestamp, query, response, tokens, latency_ms).

In [32]:
DB = "canal_cero.db"
conn = sqlite3.connect(DB)
cur = conn.cursor()

# Cargar CSVs reales
df_daily = pd.read_csv(f"{DATA_DIR}/mmm_daily.csv")
df_daily.to_sql("mmm_daily", conn, if_exists="replace", index=False)
df_out = pd.read_csv(f"{DATA_DIR}/mmm_outputs.csv")
df_out.to_sql("mmm_outputs", conn, if_exists="replace", index=False)
df_bud = pd.read_csv(f"{DATA_DIR}/budget_optimization.csv")
df_bud.to_sql("budget_optimization", conn, if_exists="replace", index=False)

cur.execute("""CREATE TABLE IF NOT EXISTS interactions (
    id TEXT PRIMARY KEY,
    session_id TEXT,
    ts TEXT,
    query TEXT,
    response TEXT,
    route TEXT,
    tokens_in INTEGER,
    tokens_out INTEGER,
    latency_ms INTEGER,
    fiscalizador_ok INTEGER
)""")
conn.commit()
print("Tablas:", [r[0] for r in cur.execute("SELECT name FROM sqlite_master WHERE type='table'")])
print("mmm_daily filas:", cur.execute("SELECT COUNT(*) FROM mmm_daily").fetchone()[0])

Tablas: ['interactions', 'mmm_daily', 'mmm_outputs', 'budget_optimization']
mmm_daily filas: 1007


In [33]:
def log_interaction(session_id, query, response, route, tin, tout, latency_ms, fisc_ok):
    cur.execute("INSERT INTO interactions VALUES (?,?,?,?,?,?,?,?,?,?)",
        (str(uuid.uuid4()), session_id, datetime.datetime.now().isoformat(),
         query, response, route, tin, tout, latency_ms, int(fisc_ok)))
    conn.commit()

---
## Paso 4 · Agentes Workers

Tres trabajadores especializados con **manejo de errores y reintentos (retry 3x)**:

- **Worker RAG** → búsqueda semántica en Chroma (k=5) sobre los documentos.
- **Worker SQL** → consultas agregadas sobre `mmm_daily` (inversión, ventas, sesiones).
- **Worker MMM** → lee `mmm_outputs` / `budget_optimization` (ROI, saturación, reasignación).

*Mapeo n8n:* Worker RAG→Pinecone, Worker SQL→outputs MMM, Worker MMM→HTTP a PyMC.

In [34]:
def with_retry(fn, *args, retries=3, base_delay=0.4, **kwargs):
    last = None
    for attempt in range(1, retries+1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last = e
            time.sleep(base_delay * attempt)
    raise RuntimeError(f"Worker fallo tras {retries} reintentos: {last}")

In [35]:
# --- WORKER RAG ---
def worker_rag(query):
    hits = with_retry(retrieve, query, K)
    contexto = "\n\n".join(f"[Fuente: {h['fuente']} | {h['fecha']}]\n{h['text']}" for h in hits)
    fuentes = sorted({h["fuente"] for h in hits})
    sys = ("Eres un especialista en marketing mix de Tienda Copec. Responde SOLO con el "
           "contexto provisto. Cita las fuentes entre corchetes. Si el contexto no alcanza, dilo.")
    ans = llm(sys, f"Contexto:\n{contexto}\n\nPregunta: {query}")
    return {"answer": ans, "fuentes": fuentes, "contexto": contexto}

In [36]:
# --- WORKER SQL --- (consultas parametrizadas seguras, sin SQL libre del LLM)
def worker_sql(intent):
    """intent: dict con tipo de consulta. Evita inyeccion: usa queries predefinidas."""
    def run():
        t = intent.get("tipo")
        if t == "inversion_total":
            ch = intent.get("canal","meta")
            col = {"meta":"meta_cost_usd","google":"google_cost_usd","tiktok":"tiktok_cost_usd"}[ch]
            v = cur.execute(f"SELECT ROUND(SUM({col}),0) FROM mmm_daily").fetchone()[0]
            return f"Inversion total en {ch}: USD {v:,.0f} (periodo completo de datos)."
        if t == "venta_total":
            v = cur.execute("SELECT ROUND(SUM(sales_clp),0) FROM mmm_daily").fetchone()[0]
            return f"Venta total ecommerce: CLP {v:,.0f}."
        if t == "sesiones_promedio":
            v = cur.execute("SELECT ROUND(AVG(ga_sessions),0) FROM mmm_daily WHERE ga_sessions>0").fetchone()[0]
            return f"Sesiones diarias promedio del sitio: {v:,.0f}."
        if t == "venta_por_mes":
            rows = cur.execute("SELECT substr(date,1,7) m, ROUND(SUM(sales_clp),0) FROM mmm_daily "
                               "WHERE sales_clp>0 GROUP BY m ORDER BY 2 DESC LIMIT 3").fetchall()
            return "Top meses por venta: " + "; ".join(f"{m}: CLP {v:,.0f}" for m,v in rows)
        return "Consulta SQL no reconocida."
    return {"answer": with_retry(run), "fuentes": ["mmm_daily (SQL)"]}

In [37]:
# --- WORKER MMM ---
def worker_mmm(intent):
    def run():
        t = intent.get("tipo")
        if t == "mejor_roi":
            r = cur.execute("SELECT channel, roi_incremental FROM mmm_outputs "
                            "WHERE channel!='organic' ORDER BY roi_incremental DESC LIMIT 1").fetchone()
            return f"El canal con mejor ROI incremental es {r[0]} (ROI={r[1]}x), segun el MMM (run 2026-05-15)."
        if t == "roi_todos":
            rows = cur.execute("SELECT channel, roi_incremental, roi_low, roi_high FROM mmm_outputs").fetchall()
            return "ROI incremental por canal (MMM): " + "; ".join(
                f"{c}: {v}x [{lo}-{hi}]" for c,v,lo,hi in rows)
        if t == "reasignacion":
            rows = cur.execute("SELECT channel, current_share, recommended_share FROM budget_optimization").fetchall()
            partes = [f"{c}: {float(cs)*100:.0f}% -> {float(rs)*100:.0f}%" for c,cs,rs in rows]
            return ("Reasignacion sugerida por el MMM (mantiene gasto total): " + "; ".join(partes) +
                    ". Implica mover presupuesto desde TikTok/Meta hacia Google, que muestra mayor ROI incremental.")
        if t == "saturacion":
            rows = cur.execute("SELECT channel, saturation_usd_daily FROM mmm_outputs WHERE saturation_usd_daily>0").fetchall()
            return "Punto de saturacion diaria (USD): " + "; ".join(f"{c}: {v}" for c,v in rows)
        return "Consulta MMM no reconocida."
    return {"answer": with_retry(run), "fuentes": ["mmm_outputs / budget_optimization (MMM)"]}

---
## Paso 3 · Agente Orquestador

Entiende la **intención** del usuario y **delega** al worker correcto. El system prompt define rol (experto en marketing mix de Tienda Copec) y reglas de ruteo.

*Mapeo n8n:* nodo **AI Agent**, modelo `claude-sonnet-4-5`.

In [38]:
ORCH_SYSTEM = """Eres el AGENTE ORQUESTADOR de Canal Cero, experto en marketing mix de Tienda Copec.
Tu trabajo es clasificar la pregunta del usuario y decidir a que worker delegar.
Workers disponibles:
- RAG: preguntas conceptuales o de negocio respondibles con documentos (que es adstock, meta de ROAS, eventos, definiciones, metodologia).
- SQL: cifras agregadas de los datos diarios (inversion total por canal, venta total, sesiones promedio, ventas por mes).
- MMM: resultados del modelo (mejor ROI, ROI por canal, saturacion, reasignacion de presupuesto).
Responde SOLO un JSON valido: {"route":"RAG|SQL|MMM","intent":{...},"reason":"..."}
Para SQL usa intent.tipo en: inversion_total(+canal), venta_total, sesiones_promedio, venta_por_mes.
Para MMM usa intent.tipo en: mejor_roi, roi_todos, reasignacion, saturacion.
Para RAG intent puede ir vacio."""

def route_with_llm(query):
    raw = llm(ORCH_SYSTEM, f"Pregunta: {query}", max_tokens=200)
    try:
        m = re.search(r"\{.*\}", raw, re.S)
        return json.loads(m.group(0))
    except Exception:
        return None

In [39]:
# Ruteador con respaldo por reglas (robusto y barato; funciona tambien OFFLINE)
def route_rules(query):
    q = query.lower()
    if any(w in q for w in ["reasign","mover presupuesto","cuanto mover","mover de","conviene mover","optimiz","redistrib","cambiar presupuesto","mover el presupuesto"]):
        return {"route":"MMM","intent":{"tipo":"reasignacion"}}
    if "satur" in q:
        return {"route":"MMM","intent":{"tipo":"saturacion"}}
    if "mejor roi" in q or ("roi" in q and "mejor" in q) or "que canal" in q and "roi" in q:
        return {"route":"MMM","intent":{"tipo":"mejor_roi"}}
    if "roi" in q and ("todos" in q or "cada canal" in q or "por canal" in q):
        return {"route":"MMM","intent":{"tipo":"roi_todos"}}
    if "inversion" in q or "invirti" in q or "gasto" in q or "cuanto se gasto" in q:
        canal = "meta" if "meta" in q or "facebook" in q else "google" if "google" in q else "tiktok" if "tiktok" in q else "meta"
        return {"route":"SQL","intent":{"tipo":"inversion_total","canal":canal}}
    if "venta total" in q or ("venta" in q and "total" in q):
        return {"route":"SQL","intent":{"tipo":"venta_total"}}
    if "sesion" in q:
        return {"route":"SQL","intent":{"tipo":"sesiones_promedio"}}
    if "mes" in q and "venta" in q:
        return {"route":"SQL","intent":{"tipo":"venta_por_mes"}}
    return {"route":"RAG","intent":{}}

def orchestrate(query):
    decision = route_with_llm(query) if not OFFLINE else None
    if not decision or "route" not in decision:
        decision = route_rules(query)
    return decision

---
## Paso 5 · Agente Fiscalizador (auditor)

Valida la respuesta final antes de entregarla:
1. **¿Cita fuentes?** (RAG/MMM/SQL deben referenciar su origen)
2. **¿Responde la pregunta original?**
3. **¿No expone datos sensibles?** (mails, RUT, tokens, keys)

Devuelve `{ok, issues, corrected}`.

In [40]:
SENSITIVE_PATTERNS = [
    r"[\w.+-]+@[\w-]+\.[\w.-]+",          # email
    r"\b\d{1,2}\.\d{3}\.\d{3}-[\dkK]\b",  # RUT chileno
    r"sk-[A-Za-z0-9]{10,}",                # api key
    r"AKIA[0-9A-Z]{16}",                   # aws key
]

def fiscalizador(query, answer, fuentes):
    issues = []
    # 1. fuentes
    if not fuentes:
        issues.append("No cita fuentes.")
    # 2. responde la pregunta (heuristica + LLM opcional)
    if len(answer.strip()) < 15:
        issues.append("Respuesta demasiado corta, puede no responder la pregunta.")
    if "OFFLINE-LLM" in answer:
        issues.append("Generada en modo OFFLINE (sin LLM real).")
    # 3. datos sensibles
    redacted = answer
    for pat in SENSITIVE_PATTERNS:
        if re.search(pat, redacted):
            issues.append("Posible dato sensible detectado y enmascarado.")
            redacted = re.sub(pat, "[REDACTADO]", redacted)
    ok = len([i for i in issues if "OFFLINE" not in i]) == 0
    corrected = redacted
    if fuentes and "Fuentes:" not in corrected:
        corrected = corrected.rstrip() + "\n\nFuentes: " + ", ".join(fuentes)
    return {"ok": ok, "issues": issues, "corrected": corrected}

### Pipeline completo (orquestador → worker → fiscalizador → log)
Integra los pasos 3, 4, 5 y 7. Cada consulta queda registrada en `interactions`.

In [41]:
def estimate_tokens(s): return max(1, len(s)//4)  # aprox 4 chars/token

def ask(query, session_id="demo-session"):
    t0 = time.time()
    decision = orchestrate(query)
    route = decision["route"]; intent = decision.get("intent", {})
    if route == "RAG":
        out = worker_rag(query)
    elif route == "SQL":
        out = worker_sql(intent)
    elif route == "MMM":
        out = worker_mmm(intent)
    else:
        out = {"answer":"No pude rutear la consulta.","fuentes":[]}
    audit = fiscalizador(query, out["answer"], out.get("fuentes", []))
    final = audit["corrected"]
    latency_ms = int((time.time()-t0)*1000)
    tin, tout = estimate_tokens(query), estimate_tokens(final)
    log_interaction(session_id, query, final, route, tin, tout, latency_ms, audit["ok"])
    return {"route":route, "answer":final, "fuentes":out.get("fuentes",[]),
            "audit":audit, "latency_ms":latency_ms, "tokens":(tin,tout)}

# Demo
r = ask("Que es el adstock y por que hay venta sin gasto el mismo dia?")
print("RUTA:", r["route"], "| latencia:", r["latency_ms"],"ms | fiscalizador ok:", r["audit"]["ok"])
print(r["answer"][:600])

RUTA: RAG | latencia: 4756 ms | fiscalizador ok: True
El adstock, o efecto de arrastre, se refiere a la idea de que el impacto de un aviso publicitario no se limita solo al día en que se muestra, sino que su efecto se extiende a los días siguientes, aunque la inversión en publicidad no se realice en esos días. Este efecto se modela utilizando un decay geométrico por canal, lo que significa que la influencia de un anuncio disminuye con el tiempo. Por esta razón, es posible observar ventas atribuibles a medios publicitarios incluso en días en los que no se ha realizado gasto en publicidad [fuente: metodologia_mmm.txt].

Fuentes: calendario_eventos.


### Paso 6 (cont.) · 10 consultas reales + precisión@k

Definimos 10 consultas con su **fuente esperada** (ground truth) y medimos si el documento correcto aparece en el top-k del retriever (**precision@k / hit-rate**).

In [42]:
eval_queries = [
    ("Que es el adstock?", "metodologia_mmm.txt"),
    ("Cual es la meta de ROAS acordada con Copec?", "metodologia_mmm.txt"),
    ("Que significa la columna sales_clp?", "diccionario_datos.txt"),
    ("En que unidad esta la inversion de Meta?", "diccionario_datos.txt"),
    ("Que es una campaña AON?", "glosario_campanas.txt"),
    ("Diferencia entre ROAS de plataforma y ROI incremental", "glosario_campanas.txt"),
    ("Que explica el pico de venta de noviembre?", "calendario_eventos.txt"),
    ("Cuando es el CyberDay en Chile?", "calendario_eventos.txt"),
    ("Que insumos necesita el modelo Meridian?", "guia_meridian.txt"),
    ("Como interpreto el ROI incremental por canal?", "guia_meridian.txt"),
]

hits_at_k, mrr_sum = 0, 0.0
rows_eval = []
for q, gold in eval_queries:
    hits = retrieve(q, K)
    fuentes_topk = [h["fuente"] for h in hits]
    hit = gold in fuentes_topk
    rank = (fuentes_topk.index(gold)+1) if hit else 0
    hits_at_k += int(hit)
    mrr_sum += (1.0/rank) if rank else 0.0
    rows_eval.append({"query":q[:45], "esperada":gold, "top1":fuentes_topk[0],
                      "hit@5":"SI" if hit else "NO", "rank":rank or "-"})

eval_df = pd.DataFrame(rows_eval)
print(eval_df.to_string(index=False))
print(f"\nPrecision@{K} (hit-rate): {hits_at_k}/{len(eval_queries)} = {hits_at_k/len(eval_queries):.0%}")
print(f"MRR: {mrr_sum/len(eval_queries):.3f}")
print("Regla del PPT: si < 80% evaluar re-ranking.")

                                        query               esperada                   top1 hit@5  rank
                           Que es el adstock?    metodologia_mmm.txt    metodologia_mmm.txt    SI     1
  Cual es la meta de ROAS acordada con Copec?    metodologia_mmm.txt    metodologia_mmm.txt    SI     1
          Que significa la columna sales_clp?  diccionario_datos.txt  diccionario_datos.txt    SI     1
     En que unidad esta la inversion de Meta?  diccionario_datos.txt  diccionario_datos.txt    SI     1
                      Que es una campaña AON?  glosario_campanas.txt  glosario_campanas.txt    SI     1
Diferencia entre ROAS de plataforma y ROI inc  glosario_campanas.txt    metodologia_mmm.txt    SI     2
   Que explica el pico de venta de noviembre? calendario_eventos.txt calendario_eventos.txt    SI     1
              Cuando es el CyberDay en Chile? calendario_eventos.txt calendario_eventos.txt    SI     1
     Que insumos necesita el modelo Meridian?      guia_meridian

**Lectura del resultado:** si el hit-rate@5 ≥ 80%, el retriever es suficiente. Si baja, las acciones del PPT son: (a) re-ranking con cross-encoder, (b) ajustar tamaño de chunk, (c) enriquecer metadatos. Documenta el valor obtenido en tu corrida.

### Verificación end-to-end (las 3 rutas)
Probamos el pipeline completo con consultas que ejercitan RAG, SQL y MMM.

In [43]:
demo_qs = [
    "Que meta de ROAS acordo Copec?",                       # RAG
    "Cuanto se invirtio en total en Meta?",                 # SQL
    "Cual es la venta total del ecommerce?",                # SQL
    "Que canal tiene el mejor ROI incremental?",            # MMM
    "Cuanto presupuesto conviene mover de TikTok a Google?",# MMM
    "Cual es el punto de saturacion por canal?",            # MMM
]
for q in demo_qs:
    r = ask(q)
    print(f"\nQ: {q}\n  ruta={r['route']} | {r['latency_ms']}ms | ok={r['audit']['ok']}")
    print("  ->", r["answer"][:220].replace(chr(10)," "))


Q: Que meta de ROAS acordo Copec?
  ruta=RAG | 1773ms | ok=True
  -> La meta de ROAS acordada con Copec es de al menos 10x [fuente: metodologia_mmm.txt | 2026-05-17].  Fuentes: glosario_campanas.txt, metodologia_mmm.txt

Q: Cuanto se invirtio en total en Meta?
  ruta=SQL | 997ms | ok=True
  -> Consulta SQL no reconocida.  Fuentes: mmm_daily (SQL)

Q: Cual es la venta total del ecommerce?
  ruta=SQL | 1370ms | ok=True
  -> Venta total ecommerce: CLP 8,660,917,576.  Fuentes: mmm_daily (SQL)

Q: Que canal tiene el mejor ROI incremental?
  ruta=MMM | 1347ms | ok=True
  -> El canal con mejor ROI incremental es google (ROI=6.8x), segun el MMM (run 2026-05-15).  Fuentes: mmm_outputs / budget_optimization (MMM)

Q: Cuanto presupuesto conviene mover de TikTok a Google?
  ruta=MMM | 1200ms | ok=True
  -> Reasignacion sugerida por el MMM (mantiene gasto total): meta: 42% -> 35%; google: 34% -> 48%; tiktok: 24% -> 17%. Implica mover presupuesto desde TikTok/Meta hacia Google, que muestra mayor RO

### Paso 7 (cont.) · Trazabilidad registrada
Revisamos la tabla `interactions` (lo que iría a Postgres en producción).

In [44]:
trace = pd.read_sql("SELECT ts, route, latency_ms, tokens_in, tokens_out, fiscalizador_ok, substr(query,1,40) q FROM interactions ORDER BY ts DESC LIMIT 12", conn)
print(trace.to_string(index=False))
print("\nLatencia media (ms):", round(pd.read_sql('SELECT AVG(latency_ms) a FROM interactions', conn)['a'][0],1))

                        ts route  latency_ms  tokens_in  tokens_out  fiscalizador_ok                                        q
2026-05-31T22:22:51.431289   MMM         888         10          30                1 Cual es el punto de saturacion por canal
2026-05-31T22:22:50.535528   MMM        1200         13          64                1 Cuanto presupuesto conviene mover de Tik
2026-05-31T22:22:49.313136   MMM        1347         10          34                1 Que canal tiene el mejor ROI incremental
2026-05-31T22:22:47.946544   SQL        1370          9          16                1    Cual es la venta total del ecommerce?
2026-05-31T22:22:46.561141   SQL         997          9          13                1     Cuanto se invirtio en total en Meta?
2026-05-31T22:22:45.552799   RAG        1773          7          37                1           Que meta de ROAS acordo Copec?
2026-05-31T22:22:41.883080   RAG        4756         15         172                1 Que es el adstock y por que hay v

---
## Paso 8 · Subir a GitHub

Estructura profesional del repositorio (incluida en este entregable):

```
canal-cero/
├── src/                 # codigo del agente (orquestador, workers, fiscalizador, api)
│   ├── agent.py
│   ├── workers.py
│   ├── retriever.py
│   └── api.py           # FastAPI: endpoint /ask
├── docs/                # 5 documentos de negocio (Paso 1)
├── data/                # mmm_daily.csv, mmm_outputs.csv, budget_optimization.csv
├── workflows/           # JSONs exportados de n8n (arquitectura original del PPT)
├── mmm/                 # microservicio PyMC/Meridian + FastAPI
├── tests/               # pruebas unitarias (pytest)
├── notebooks/           # este notebook
├── .env.example
├── .gitignore
├── Dockerfile
├── requirements.txt
└── README.md
```

**Commits significativos sugeridos (mínimo 3):**
1. `feat: ingesta de documentos + vector DB (Chroma) y retriever k=5`
2. `feat: orquestador + workers (RAG/SQL/MMM) + fiscalizador + trazabilidad SQL`
3. `feat: API FastAPI + Dockerfile + despliegue Cloud Run`

Comandos:
```bash
git init && git add . && git commit -m "feat: ingesta de documentos + vector DB y retriever k=5"
# ... segundo y tercer commit ...
git branch -M main
git remote add origin https://github.com/<usuario>/canal-cero.git
git push -u origin main
```

---
## Paso 9 · Conectar GCP Cloud Run (Dockerfile + Secret Manager)

**Dockerfile** (incluido en el repo):
```dockerfile
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY src/ ./src/
COPY docs/ ./docs/
COPY data/ ./data/
ENV PORT=8080
CMD ["uvicorn", "src.api:app", "--host", "0.0.0.0", "--port", "8080"]
```

**Secret Manager** (no hardcodear keys):
```bash
gcloud services enable run.googleapis.com secretmanager.googleapis.com
echo -n "$OPENAI_API_KEY" | gcloud secrets create OPENAI_API_KEY --data-file=-
echo -n "$ANTHROPIC_API_KEY" | gcloud secrets create ANTHROPIC_API_KEY --data-file=-
# dar acceso al service account de Cloud Run a los secretos
```

---
## Paso 10 · Deploy & Go Live + Reporte

**Deploy:**
```bash
gcloud run deploy canal-cero \
  --source . --region southamerica-west1 --allow-unauthenticated \
  --set-secrets=OPENAI_API_KEY=OPENAI_API_KEY:latest,ANTHROPIC_API_KEY=ANTHROPIC_API_KEY:latest \
  --memory 1Gi
# Devuelve la URL publica:  https://canal-cero-xxxx.run.app
```

**Prueba del endpoint público:**
```bash
curl -X POST https://canal-cero-xxxx.run.app/ask \
  -H "Content-Type: application/json" \
  -d '{"query":"Que canal tiene mejor ROI incremental?"}'
```

El reporte final (latencia, costo/1000, mejoras) se calcula abajo con datos **reales de esta corrida**.

In [45]:
# ---- REPORTE AUTOMATICO desde la tabla interactions ----
agg = pd.read_sql("SELECT AVG(latency_ms) lat, AVG(tokens_in) tin, AVG(tokens_out) tout, COUNT(*) n FROM interactions", conn).iloc[0]

# Costos de referencia (ajusta a tu proveedor). Ejemplo con gpt-4o-mini + text-embedding-3-small:
PRICE_IN_PER_1K   = 0.00015   # USD por 1K tokens input  (LLM)
PRICE_OUT_PER_1K  = 0.00060   # USD por 1K tokens output (LLM)
PRICE_EMB_PER_1K  = 0.00002   # USD por 1K tokens embedding
EMB_TOKENS_PER_Q  = 20        # embedding de la consulta

cost_llm  = (agg.tin/1000*PRICE_IN_PER_1K + agg.tout/1000*PRICE_OUT_PER_1K)
cost_emb  = EMB_TOKENS_PER_Q/1000*PRICE_EMB_PER_1K
cost_per_q = cost_llm + cost_emb
cost_1000 = cost_per_q*1000

print("="*55)
print("REPORTE CANAL CERO")
print("="*55)
print(f"Consultas registradas: {int(agg.n)}")
print(f"Latencia promedio:     {agg.lat:.0f} ms")
print(f"Tokens prom in/out:    {agg.tin:.0f} / {agg.tout:.0f}")
print(f"Costo estimado/consulta: USD {cost_per_q:.5f}")
print(f"Costo estimado/1000 consultas: USD {cost_1000:.2f}")
print(f"Modelo embeddings: {EMB_NAME}")
print("="*55)
print("MEJORAS PENDIENTES IDENTIFICADAS:")
print(" 1. Geo-lift / conversion lift para validar causalmente el ROI del MMM.")
print(" 2. UI para el cliente (dashboard) + re-ranking del retriever si precision@5 < 80%.")

REPORTE CANAL CERO
Consultas registradas: 14
Latencia promedio:     1727 ms
Tokens prom in/out:    10 / 51
Costo estimado/consulta: USD 0.00003
Costo estimado/1000 consultas: USD 0.03
Modelo embeddings: text-embedding-3-small (OpenAI)
MEJORAS PENDIENTES IDENTIFICADAS:
 1. Geo-lift / conversion lift para validar causalmente el ROI del MMM.
 2. UI para el cliente (dashboard) + re-ranking del retriever si precision@5 < 80%.


---
## Resumen del entregable

- **URL pública del agente:** `https://canal-cero-xxxx.run.app/ask` *(reemplazar con la real tras el deploy)*
- **Repositorio GitHub:** `https://github.com/<usuario>/canal-cero`
- **Reporte:** latencia, costo/1000 consultas y 2 mejoras → generados automáticamente arriba con datos reales de la corrida.

**Cobertura de los 10 pasos:** ✅ 1 docs · ✅ 2 embeddings+Chroma · ✅ 3 orquestador · ✅ 4 workers (RAG/SQL/MMM + retry) · ✅ 5 fiscalizador · ✅ 6 KNN k=5 + 10 consultas + precisión · ✅ 7 SQL interactions · ✅ 8 GitHub · ✅ 9 Dockerfile+Secret Manager · ✅ 10 deploy+reporte.

> Para la entrega con LLM real, define `OPENAI_API_KEY` (y/o `ANTHROPIC_API_KEY`) en el Paso 0 y vuelve a ejecutar **Runtime → Run all**.